# Release-zone huck validation

**Decision:** can coaches use the highest-completion field box as a dependable individual strength?

**Assessment: needs revision as a player-strength claim.** The primary chronological check does not support that recommendation. Attempt-location tendencies repeat more, but the simple models did not establish individual archetypes beyond the league location pattern.

Source: October 12, 2026 database CSV snapshot, `src/data.json`, 2026 regular season. Named throwers, non-stall passes; hucks require forward gain >30 yards AND endpoint distance >40 yards. Source has throws for 131 of 132 listed regular-season games. No live database query.

Primary cutoff: June 15. Choose each player's highest earlier completion zone among zones with at least 5 earlier hucks; require at least two eligible zones. Ties: attempts, then zone ID. Compare later attempts there with the same player's other zones. June 22 and July 1 are overlapping sensitivity checks, not independent replications. This is exploratory work on an already viewed season, not preregistered or external validation.


In [1]:
from pathlib import Path
import json, shutil, sys
ROOT=Path.cwd()
if not (ROOT/'src/data.json').exists(): ROOT=ROOT.parent
assert (ROOT/'src/data.json').exists()
sys.path.insert(0,str(ROOT/'analysis'))
from release_zone_validation import run
NODE=shutil.which('node') or str(Path.home()/'.cache/codex-runtimes/codex-primary-runtime/dependencies/node/bin/node')
result=run(NODE)
print(json.dumps({'source':result['source'],'coverage':result['coverage']},indent=2))


{
  "source": {
    "path": "src/data.json",
    "sha256": "d137de62f252a78102e600e7076cffd70e26d335a12abb973c94613dfb0f71c9",
    "eventRowsSha256": "5c824fc5b4be360d91d16a9d9426071929c73cef301afcd1de062d0b6d651dba",
    "exportedAt": "2026-10-12T15:31:26.951331+00:00",
    "rawEvents": 78847,
    "regularEvents": 72663,
    "tables": [
      "prod.throws",
      "prod.games"
    ],
    "caveats": [
      "131 of 132 listed regular-season games have throws; Pittsburgh at Chicago on July 18 has none.",
      "The CSV export date is October 12, 2026; this is not a live connection. Counts were checked locally, not against a fresh database COUNT query.",
      "The full local exports contain other seasons; this dashboard deliberately uses only 2026.",
      "Repeated 2026 roster IDs are collapsed into one display identity; one blank roster ID is omitted from the display directory. Active flags are not used to exclude historical passes.",
      "The reference is a July 10 snapshot. These r

## Earlier selection versus later performance

Pooled rates sum completions and attempts. Equal-player gaps weight each player equally; tiny cells can make these differ from pooled gaps. Intervals use 10,000 whole-player bootstrap draws, conditional on the earlier selection. They do not remove opponent confounding or prove causality.


In [2]:
for s in result['studies']:
 b=s['bestSuccessZone']
 print(s['cutoff'], '| selected:',b['eligiblePlayers'],'| testable:',b['evaluatedPlayers'])
 for period in ['early','later']:
  a,o=b[period+'Chosen'],b[period+'Other']
  print(period, 'chosen:',a,'other:',o,'equal-player gap:',b[period+'Advantage'])
 print('Later higher / tied / lower:',b['laterPositive'],b['laterZero'],b['evaluatedPlayers']-b['laterPositive']-b['laterZero'])
print('All primary candidates:')
for r in result['studies'][0]['bestSuccessZone']['rows']:
 print(r['name'],r['zone'],'early:',r['early'],'later:',r['later'])


2026-06-15 | selected: 15 | testable: 14
early chosen: {'attempts': 86, 'successes': 67, 'rate': 0.7790697674418605} other: {'attempts': 213, 'successes': 134, 'rate': 0.6291079812206573} equal-player gap: {'mean': 0.16303502262148878, 'ci95': [0.0869971804511278, 0.24417790960084193], 'players': 14}
later chosen: {'attempts': 44, 'successes': 30, 'rate': 0.6818181818181818} other: {'attempts': 117, 'successes': 83, 'rate': 0.7094017094017094} equal-player gap: {'mean': -0.1664198546551488, 'ci95': [-0.38575628619746266, 0.02572905036140326], 'players': 14}
Later higher / tied / lower: 4 3 7
2026-06-22 | selected: 17 | testable: 13
early chosen: {'attempts': 101, 'successes': 83, 'rate': 0.8217821782178217} other: {'attempts': 221, 'successes': 147, 'rate': 0.665158371040724} equal-player gap: {'mean': 0.15564630330110207, 'ci95': [0.09035254150734025, 0.22593886898705004], 'players': 13}
later chosen: {'attempts': 39, 'successes': 27, 'rate': 0.6923076923076923} other: {'attempts': 10

## Predictive check and alternative question

Require 10 earlier hucks. Fixed models: smoothed player baseline; add a league-wide zone log-odds effect; then add a player-zone rate shrunk toward that prediction with 20 prior observations. Brier error is mean squared probability error; lower is better. Paired differences resample whole later games. These are simple baselines, not an optimized machine-learning study; opponent, weather and receiver effects are not fully controlled. Shrinkage weights 5, 10 and 40 are sensitivity checks, not tuning against later outcomes.

Completion: adding individual zone history worsened the primary shared-location model and did so for every tested cutoff and shrinkage strength. This fails to establish useful individual strengths; it does not prove none exist.

Alternative: huck attempts divided by **all located throws**. Choose an earlier high-attempt-rate zone from at least two zones with 20 throws and 3 hucks each. 18 of 27 testable players still attempted hucks more often from that zone later. Pooled later rates: 60/432 (13.9%) versus 187/2,626 (7.1%). However, adding individual zone histories did not improve the general location model. Restricting to y=20–60 weakens even the general location evidence. Do not interpret broad field-position constraints as validated player archetypes.


In [3]:
for s in result['studies']:
 print('Cutoff:',s['cutoff'])
 for task in ['successPrediction','choicePrediction']:
  print(task,json.dumps(s[task],indent=2))
 print('Personal minus shared completion error, prior sensitivity:',{k:v['personalMinusShared']['mean'] for k,v in s['priorSensitivity'].items()})
b=result['studies'][0]['bestChoiceZone']
print('Alternative:',json.dumps({k:v for k,v in b.items() if k!='rows'},indent=2))
for s in result['spatialSensitivity']:
 print(s['scope'],s['choicePrediction'],s['bestChoiceZone']['laterAdvantage'])


Cutoff: 2026-06-15
successPrediction {
  "eligiblePlayers": 61,
  "evaluatedPlayers": 55,
  "testEvents": 421,
  "testSuccesses": 283,
  "brier": {
    "player": 0.22536946831035312,
    "sharedZone": 0.223153985709449,
    "personalZone": 0.22726266750422838
  },
  "sharedMinusPlayer": {
    "mean": -0.0022154826009041697,
    "ci95": [
      -0.0056765265142093325,
      0.001575775532099502
    ],
    "games": 48
  },
  "personalMinusShared": {
    "mean": 0.004108681794779406,
    "ci95": [
      0.0016239181979490905,
      0.007054857649064742
    ],
    "games": 48
  },
  "personalMinusPlayer": {
    "mean": 0.0018931991938752352,
    "ci95": [
      -0.0024095489175104774,
      0.006802774860255137
    ],
    "games": 48
  },
  "cellPriorAttempts": 20
}
choicePrediction {
  "eligiblePlayers": 61,
  "evaluatedPlayers": 57,
  "testEvents": 5313,
  "testSuccesses": 421,
  "brier": {
    "player": 0.07146578238574869,
    "sharedZone": 0.06980919222574973,
    "personalZone": 0.07

## Difficulty check

Completion was 66.7% for 40–50-yard hucks and 54.3% beyond 70 yards. This is descriptive, not a causal estimate. The selected zones' later hucks were shorter on average than the other zones', so longer throws do not explain away their lost advantage.

Supplied `cp` has complete coverage, but model features, training dates and temporal availability have not been verified; it is not an independent past-only predictor here. Stored `cpoe` differs from `complete - cp` on 99 hucks, all recorded drops. A descriptive `complete - cp` comparison also does not rescue the selected-zone advantage. Receiver movement at release and defensive pressure are unmeasured.


In [4]:
print('Distance bands:',json.dumps(result['distanceBands'],indent=2))
print('Selected versus other zones:',json.dumps(result['difficultyComparison'],indent=2))
print('Supplied metric audit:',result['cpAudit'])


Distance bands: [
  {
    "lowerExclusive": 40,
    "upperInclusive": 50,
    "attempts": 1112,
    "successes": 742,
    "rate": 0.6672661870503597
  },
  {
    "lowerExclusive": 50,
    "upperInclusive": 60,
    "attempts": 1102,
    "successes": 712,
    "rate": 0.6460980036297641
  },
  {
    "lowerExclusive": 60,
    "upperInclusive": 70,
    "attempts": 798,
    "successes": 486,
    "rate": 0.6090225563909775
  },
  {
    "lowerExclusive": 70,
    "upperInclusive": null,
    "attempts": 495,
    "successes": 269,
    "rate": 0.5434343434343434
  }
]
Selected versus other zones: [
  {
    "period": "early",
    "chosen": true,
    "attempts": 86,
    "successes": 67,
    "rate": 0.7790697674418605,
    "meanDistance": 58.11877045990706,
    "meanSuppliedCp": 0.6418160464595212,
    "meanOutcomeMinusCp": 0.13725372098233915
  },
  {
    "period": "early",
    "chosen": false,
    "attempts": 213,
    "successes": 134,
    "rate": 0.6291079812206573,
    "meanDistance": 59.45930421

## Independent verification

Recompute geometry and the primary comparison directly from source rows using pandas/numpy, independently of the app's helpers. Fixed assertions apply to this snapshot; a changed snapshot requires reviewing the findings.


In [5]:
import pandas as pd
import numpy as np
d=pd.DataFrame(json.loads((ROOT/'src/data.json').read_text())['queries']['events']['rows'])
h=d[(d.phase=='Regular season')&~d.stall&d.thrower.notna()&(d.thrower!='unknown')&((d.ry-d.ty)>30)&(np.hypot(d.rx-d.tx,d.ry-d.ty)>40)].copy()
assert len(h)==3507 and h.complete.sum()==2209
h['zone']=np.select([h.ty<20,h.tx<-15,h.tx>=15,h.ty<40,h.ty<60,h.ty<80],['own-endzone','','','center-init','center-build','center-attack'],default='center-red')
for side,mask in [('left',(h.ty>=20)&(h.tx<-15)),('right',(h.ty>=20)&(h.tx>=15))]:
 h.loc[mask,'zone']=np.where(h.loc[mask,'ty']<60,side+'-build',side+'-attack')
z=h[h.date<'2026-06-15'].groupby(['thrower','zone']).complete.agg(['size','sum','mean']).reset_index()
z=z[z['size']>=5];z=z[z.groupby('thrower').thrower.transform('size')>=2]
best=z.sort_values(['thrower','mean','size','zone'],ascending=[True,False,False,True]).drop_duplicates('thrower').set_index('thrower').zone.to_dict()
assert best=={r['player']:r['zone'] for r in result['studies'][0]['bestSuccessZone']['rows']}
late=h[(h.date>='2026-06-15')&h.thrower.isin(best)].copy()
late['chosen']=[r.zone==best[r.thrower] for r in late.itertuples()]
testable=set(late.groupby('thrower').chosen.nunique().loc[lambda s:s==2].index)
assert len(testable)==14
c=late[late.thrower.isin(testable)].groupby('chosen').complete.agg(['size','sum'])
assert tuple(c.loc[True])==(44,30) and tuple(c.loc[False])==(117,83)
print('Passed independent totals, zone selection and later 30/44 versus 83/117 comparison.')


Passed independent totals, zone selection and later 30/44 versus 83/117 comparison.


## Coaching judgment

Use the existing map to nominate clips for scouting, not to prescribe a receiving spot or defensive denial strategy from the highest percentage alone. The narrower promising direction is repeatable deep-throwing opportunities; player identity must add value beyond ordinary field-position effects before calling it archetyping.

More seasons or broader zones could improve sample size, but require fresh validation rather than an assumed improvement. The current data does not establish that moving a thrower changes their success.

Reproducer: `release_zone_validation.py`; exact app preparation: `release_zone_inputs.mjs`; complete results and provenance: `release_zone_validation.json`. Original website source rows and application code were not changed by this analysis.
